# Full-Arm Manipulation Planner: several caps

Copy of `full_arm_planner.ipynb`, extended from one bottle cap to K caps (two for now). Each cap k has
its own grasp polytope and its own angle αₖ. The reachability A* tracks one α interval per cap, and αₖ
only changes while the arm is in cap k's grasp polytope.

- The scene is set in section 1 by `CAP_POSES` and `OBSTACLE_POSES` (no obstacles for now). Everything
  downstream loops over them.
- C-free is regenerated for this scene (section 4) and cached in `data/cfree/` under a name that records
  the number of DOF, caps and obstacles.
- Left out from the single-cap notebook: the brute-force wrist sweep, and the MIQP/GCS
  `compute_trajectory` cells.

In [35]:
# Cell 1: Import necessary libraries

import os
import numpy as np
import time
from pydrake.all import (
    DiagramBuilder, AddMultibodyPlantSceneGraph, Parser,
    Role, MeshcatVisualizer, StartMeshcat, RationalForwardKinematics, CspaceFreePolytope,
    SeparatingPlaneOrder, Rgba, RollPitchYaw,
    LinearEqualityConstraint, Sphere, Parallelism, AddDefaultVisualization,
    ConnectPlanarSceneGraphVisualizer, IrisFromCliqueCoverOptions,
    CommonSampledIrisOptions, IrisZoOptions, IrisNp2Options,
    IrisInConfigurationSpaceFromCliqueCover, RandomGenerator, RobotDiagramBuilder,
    SceneGraphCollisionChecker, MultibodyPlant, SceneGraph,
    SolverOptions, CommonSolverOption, GeometrySet, ScsSolver
)
from pydrake.geometry.optimization import GraphOfConvexSetsOptions, HPolyhedron, VPolytope, Point, Hyperellipsoid
from pydrake.geometry.optimization import ConvexHull as DrakeConvexHull
from pydrake.planning import GcsTrajectoryOptimization
from pydrake.solvers import MathematicalProgram, Solve, MosekSolver
from pydrake.trajectories import CompositeTrajectory
from pydrake.common import FindResourceOrThrow
from scipy.spatial import ConvexHull
from pydrake.math import RigidTransform, RotationMatrix
from pydrake.multibody.inverse_kinematics import InverseKinematics
from pydrake.solvers import Solve, SnoptSolver, IpoptSolver, OsqpSolver, ScsSolver
from functools import partial
from dataclasses import dataclass
import matplotlib.pyplot as plt
from ipywidgets import widgets

from pathlib import Path
import sys

def _find_repo_root(start: Path, markers=(".git", "README.md")) -> Path:
    for candidate in (start, *start.parents):
        if any((candidate / m).exists() for m in markers):
            return candidate
    raise RuntimeError(f"Could not locate repo root above {start}")

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (REPO_ROOT, REPO_ROOT / "algorithms", REPO_ROOT / "algorithms" / "nlp_sampling"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from ciris_plant_visualizer import CIrisPlantVisualizer

## 1. Load the full Panda arm + caps + obstacles

Same URDFs, welds and load order as `full_arm_planner.ipynb`: the arm and hand come first, so their
position indices do not change. Caps and obstacles are added from `CAP_POSES` / `OBSTACLE_POSES`. Each cap
adds one DOF (its `cap_to_base` joint) at the end of q, so cap k's angle αₖ is `q[idx_caps[k]]`.

In [36]:
print("Setting up the plant and scene graph...")

builder = RobotDiagramBuilder(time_step=0.0)
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = Parser(plant, scene_graph)
parser.SetAutoRenaming(True)

Setting up the plant and scene graph...


In [37]:
print("Loading Panda robot models...")

# panda_arm  = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_arm.urdf")[0]
panda_arm = parser.AddModels(url="file:///home/julialopezgomez/optimisation-based-manipulation-planner/my_sdfs/panda_arm.urdf")[0]
panda_hand = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_hand.urdf")[0]

print("Welding Panda arm and hand...")

plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("panda_link0", panda_arm),
    RigidTransform())

X_8H = RigidTransform(RollPitchYaw(0.0, 0.0, -np.deg2rad(45.0)), [0.0, 0.0, 0.0])
plant.WeldFrames(
    plant.GetFrameByName("panda_link8", panda_arm),
    plant.GetFrameByName("panda_hand", panda_hand),
    X_8H)

print("Setting default finger joint positions...")
plant.GetJointByName("panda_finger_joint1", panda_hand).set_default_translation(-0.024)

Loading Panda robot models...
Welding Panda arm and hand...
Setting default finger joint positions...


In [38]:
print("Adding the bottle caps and obstacles...")

CAP_POSES = [[0.5, 0.0, 0.0], [0.5, 0.25, 0.0]]   # cap k is welded at CAP_POSES[k]
OBSTACLE_POSES = []                                # the single-cap scene had [[0.51, 0.031, 0.01], [0.467, -0.005, 0.01]]

caps = []
for p in CAP_POSES:
    cap_k = parser.AddModels(file_name="my_sdfs/bottle_cap.sdf")[0]
    plant.WeldFrames(
        plant.world_frame(),
        plant.GetFrameByName("base_link", cap_k),
        RigidTransform(RotationMatrix(), p))
    caps.append(cap_k)

obstacles = []
for p in OBSTACLE_POSES:
    obstacle = parser.AddModels("my_sdfs/obstacle.sdf")[0]
    plant.WeldFrames(
        plant.world_frame(),
        plant.GetFrameByName("obstacle_link", obstacle),
        RigidTransform(RotationMatrix(), p))
    obstacles.append(obstacle)

K = len(caps)
print(f"{K} caps, {len(obstacles)} obstacles")

Adding the bottle caps and obstacles...
2 caps, 0 obstacles


In [39]:
print("Finalising plant...")
print("Bodies:", plant.num_bodies(), "Joints:", plant.num_joints())


plant.Finalize()

inspector = scene_graph.model_inspector()
num_frames = len(list(inspector.GetAllFrameIds()))
num_geoms = sum(inspector.NumGeometriesForFrame(fid) for fid in inspector.GetAllFrameIds())
print("Frames:", num_frames, "Geometries:", num_geoms)


print("Plant finalised.")

print("Number of positions: ", plant.num_positions())


q_star = np.zeros(plant.num_positions())

# print("Initialising CspaceFreePolytope...")
# cspace_free_polytope = CspaceFreePolytope(
#     plant,
#     scene_graph,
#     SeparatingPlaneOrder.kAffine,
#     q_star)

print("Initializing CIrisPlantVisualizer...")
visualizer = CIrisPlantVisualizer(
    plant,
    builder,
    scene_graph,
    # cspace_free_polytope,
    viz_role=Role.kIllustration,
    allow_plus_3dof=True
)

visualizer.task_space_diagram.ForcedPublish(visualizer.task_space_diagram_context)

INFO:drake:Meshcat listening for connections at http://localhost:7004


Finalising plant...
Bodies: 18 Joints: 17
Frames: 18 Geometries: 95
Plant finalised.
Number of positions:  11
Initializing CIrisPlantVisualizer...
Visualisations won't work properly. Can't visualize the TC-Space of plants with more than 3-DOF. The first 3 DOF from the plant will be visualized


In [40]:
sliders = []

plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

for i in range(plant.num_positions()):
    q_low = plant.GetPositionLowerLimits()[i]
    q_high = plant.GetPositionUpperLimits()[i]
    step = (q_high - q_low) / 100
    sliders.append(widgets.FloatSlider(
        min=q_low, max=q_high, 
        value=0, step=step, 
        description=f"q{i}"))
    
q = np.zeros(plant.num_positions())

def report_collisions(q):
    plant.SetPositions(plant_context, q)
    query_object = scene_graph.get_query_output_port().Eval(
        scene_graph.GetMyContextFromRoot(diagram_context)
    )
    inspector = scene_graph.model_inspector()

    pairs = query_object.ComputePointPairPenetration()
    if not pairs:
        print("No penetrations")
        return

    for pair in pairs:
        name_A = inspector.GetName(pair.id_A)
        name_B = inspector.GetName(pair.id_B)
        print(f"{name_A} <-> {name_B}, depth={pair.depth}")



out = widgets.Output()
display(out)

def handle_slider_change(change, idx):
    q[idx] = change["new"]
    plant.SetPositions(plant_context, q)
    diagram.ForcedPublish(diagram_context)
    with out:
        out.clear_output(wait=True)
        
        report_collisions(q)
        print(f"{visualizer.check_collision_q_by_ik(q)} \t {q}", flush=True)
    
    
idx = 0
for slider in sliders:
    slider.observe(partial(handle_slider_change, idx = idx), names='value')
    idx+=1

for slider in sliders:
    display(slider)

Output()

FloatSlider(value=0.0, description='q0', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=0.0, description='q1', max=1.7628, min=-1.7628, step=0.035255999999999996)

FloatSlider(value=0.0, description='q2', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=-0.0698, description='q3', max=-0.0698, min=-3.0718, step=0.03002)

...

FloatSlider(value=0.0, description='q6', max=1.0, min=-1.0, step=0.02)

FloatSlider(value=0.0, description='q7', max=0.0, min=-0.045, step=0.00045)

FloatSlider(value=0.0, description='q8', max=0.045, step=0.00045)

FloatSlider(value=0.0, description='q9', max=3.14, min=-3.14, step=0.06280000000000001)

FloatSlider(value=0.0, description='q10', max=3.14, min=-3.14, step=0.06280000000000001)

In [41]:
def joint_position_index(plant, joint_name, model_instance=None):
    if model_instance is None:
        joint = plant.GetJointByName(joint_name)
    else:
        joint = plant.GetJointByName(joint_name, model_instance)

    if joint.num_positions() != 1:
        raise ValueError(
            f"Joint {joint_name} has {joint.num_positions()} positions; "
            "this helper assumes a 1-DOF joint."
        )

    return joint.position_start()

for i, name in enumerate(plant.GetPositionNames()):
    print(f"{i:2d}: {name}")

idx_caps = [joint_position_index(plant, "cap_to_base", c) for c in caps]   # cap k's angle is q[idx_caps[k]]
nq = plant.num_positions()
assert nq == 9 + K, f"expected 7 arm + 2 finger + {K} cap DOF, got {nq}"
print("cap indices:", idx_caps)

 0: panda_panda_joint1_q
 1: panda_panda_joint2_q
 2: panda_panda_joint3_q
 3: panda_panda_joint4_q
 4: panda_panda_joint5_q
 5: panda_panda_joint6_q
 6: panda_panda_joint7_q
 7: panda_hand_panda_finger_joint1_x
 8: panda_hand_panda_finger_joint2_x
 9: bottle_cap_cap_to_base_q
10: bottle_cap_1_cap_to_base_q
cap indices: [9, 10]


## 2. One grasp polytope per cap (wrist-axis analysis, #80)

Same construction as the single-cap notebook, done once per cap. The grasp constraints h(q) = 0,
g(q) ≤ 0 are written for that cap's frame, a grasp seed q0ₖ is found for it, and the constraints are
linearised around q0ₖ (±`margin`), plus the grasp joint limits: wrist in [−1, 1], fingers at the grasp
width.

The grasp constraints do not depend on any cap angle (the cap frame is the welded `base_link`), so every
grasp polytope leaves **all** cap angles free. Which cap a grasp turns is not encoded in the polytope; the
A* (section 5) knows it from which polytope the arm is in.

`placement_polytope` (cell below) is only needed by the `ManipulationPlanner` constructor.

### Find a grasp configuration for each cap

`find_grasp_seed` (nlp-sampling downhill phase 1) accepts a point once its error is ≤ `good_err_tol = 0.01`,
which is loose for this constraint: depending on the random probe, it can stop 2-4 cm or ~2° off the grasp,
with arm joints pinned at their limits. The single-cap notebook's seed happened to converge to |h| ≈ 5e-4.
With an 11-D box the probes differ, and the raw seeds were at |h| ≈ 3e-2, which is visibly off the cap.

So each seed is **refined** with an NLP that uses the same h and g, from `make_grasp_constraints`:
min ‖q − q_seed‖² s.t. |h(q)| ≤ `H_TOL`, g(q) ≤ 0, the grasp joint limits, and the non-wrist arm joints at
least `LIMIT_MARGIN` inside their limits (so the ±0.05 rad trust region of section 5 is not cut off by a
joint limit). The search goes through `find_grasp_seed`'s random seeds until the refinement converges and
the grasp is collision-free.

In [42]:
from wrist_axis_grasp_polytope import find_grasp_seed
from nlp_sampling_standalone_test import make_grasp_constraints, build_autodiff_companion, make_joint_indices, make_joint_limits

plant, plant_context = visualizer.plant, visualizer.plant_context
E = plant.GetFrameByName("panda_hand", panda_hand)
idx = make_joint_indices(plant, panda_arm, panda_hand, caps[0])   # arm and finger indices are the same for every cap
idx_wrist, idx_f1, idx_f2 = idx["idx_wrist"], idx["idx_f1"], idx["idx_f2"]
lower, upper = make_joint_limits(plant, idx_wrist, idx_f1, idx_f2) # grasp limits


def make_cap_grasp_constraints(cap_instance):
    """Grasp constraints h(q) = 0, g(q) <= 0 and their Jacobians, for grasping cap_instance."""
    Cap = plant.GetFrameByName("base_link", cap_instance)
    plant_ad, plant_context_ad, E_ad, Cap_ad = build_autodiff_companion(plant, E, Cap)
    h_grasp, g_grasp, Jh, Jg = make_grasp_constraints(
        plant, plant_context,
        plant_ad, plant_context_ad,
        E, Cap,
        E_ad, Cap_ad)
    return dict(
        plant=plant,
        h=h_grasp,
        g=g_grasp,
        Jh=Jh,
        Jg=Jg,
        idx_wrist=idx_wrist,
        lower=lower,
        upper=upper,
        _diagram=visualizer.task_space_diagram,
        _diagram_context=visualizer.task_space_diagram_context,
        _plant_context=plant_context,
        _plant_ad=plant_ad,
        _plant_context_ad=plant_context_ad
    )


cap_grasp_constraints = [make_cap_grasp_constraints(c) for c in caps]

H_TOL = 1e-6          # |h(q0)| after refinement (the single-cap seed had ~5e-4)
LIMIT_MARGIN = 0.05   # rad: keep the non-wrist arm joints this far inside their limits
idx_arm_no_wrist = [i for i in idx["idx_arm"] if i != idx_wrist]


def refine_grasp_seed(gc, q_seed):
    """Project q_seed onto the grasp: min ||q - q_seed||^2 s.t. |h(q)| <= H_TOL, g(q) <= 0, grasp limits."""
    lo, hi = np.array(gc["lower"], dtype=float), np.array(gc["upper"], dtype=float)
    lo[idx_arm_no_wrist] += LIMIT_MARGIN
    hi[idx_arm_no_wrist] -= LIMIT_MARGIN
    prog = MathematicalProgram()
    q = prog.NewContinuousVariables(len(q_seed), "q")
    prog.AddBoundingBoxConstraint(lo, hi, q)
    prog.AddConstraint(gc["h"], -H_TOL * np.ones(4), H_TOL * np.ones(4), q)   # h, g accept AutoDiff
    prog.AddConstraint(gc["g"], np.full(3, -np.inf), np.zeros(3), q)
    prog.AddQuadraticErrorCost(np.eye(len(q_seed)), q_seed, q)
    result = Solve(prog, np.clip(q_seed, lo, hi))
    return result.GetSolution(q), result.is_success()


def find_cap_grasp(gc, max_seeds=20):
    """First random seed whose refined grasp converges and is collision-free."""
    for s in range(max_seeds):
        q_seed = find_grasp_seed(gc, seed=s)
        q, ok = refine_grasp_seed(gc, q_seed)
        q[idx_wrist] = -1.0
        q[idx_f1] = -0.02405
        q[idx_f2] = 0.02405
        if ok and not visualizer.check_collision_q_by_ik(q):
            return q, s
    raise RuntimeError(f"no converged, collision-free grasp in {max_seeds} seeds")


q0s = []   # q0s[k]: grasp seed for cap k
for k, gc in enumerate(cap_grasp_constraints):
    q0_k, s = find_cap_grasp(gc)
    q0s.append(q0_k)
    print(f"cap {k + 1} (seed {s}): |h(q0)| = {np.abs(gc['h'](q0_k)).max():.1e}, "
          f"g(q0) = {np.round(gc['g'](q0_k), 4)}")
    print(f"  q0 = {np.round(q0_k, 3)}")


cap 1 (seed 1): |h(q0)| = 1.0e-06, g(q0) = [-0.0003 -0.0047 -0.1   ]
  q0 = [-0.636  0.633  0.526 -2.286 -0.828  2.726 -1.    -0.024  0.024  1.572
 -1.379]
cap 2 (seed 0): |h(q0)| = 1.0e-06, g(q0) = [-0.    -0.005 -0.1  ]
  q0 = [-2.342 -0.654  2.847 -2.12   0.441  2.716 -1.    -0.024  0.024 -1.264
  1.08 ]


In [43]:
for k, q0_k in enumerate(q0s):
    plant.SetPositions(plant_context, q0_k)
    diagram.ForcedPublish(diagram_context)
    print(f"cap {k + 1}: q0 in collision:", visualizer.check_collision_q_by_ik(q0_k))
    time.sleep(3.0)

cap 1: q0 in collision: 0.0
cap 2: q0 in collision: 0.0


### Build each cap's grasp polytope in the nullspace of its grasp constraint

In [44]:
def nullspace_from_jacobian(J, rtol=1e-8):
    U, s, Vt = np.linalg.svd(J, full_matrices=True)

    tol = rtol * s[0] if len(s) else rtol
    rank = np.sum(s > tol)

    N = Vt[rank:, :].T       # (n, n-rank)
    P = N @ N.T              # (n, n)

    return N, P, s, rank


for k, (gc, q0_k) in enumerate(zip(cap_grasp_constraints, q0s)):
    Jh0 = gc["Jh"](q0_k)
    N, P, singular_values, rank = nullspace_from_jacobian(Jh0)
    print(f"cap {k + 1}: Jh {Jh0.shape}, rank {rank}, N {N.shape}")

cap 1: Jh (4, 11), rank 4, N (11, 7)
cap 2: Jh (4, 11), rank 4, N (11, 7)


In [45]:
def linearised_grasp_polytope(grasp_constraints, q0, margin=1e-3):
    """Grasp polytope of one cap: its grasp constraints linearised around q0, plus the grasp joint limits."""
    nq = len(q0)
    g0 = grasp_constraints["g"](q0)
    h0 = grasp_constraints["h"](q0)
    Jh0 = grasp_constraints["Jh"](q0)
    Jg0 = grasp_constraints["Jg"](q0)

    # ------------------------------------------------------------------
    # 1. Turn h(q) = 0 into |h(q)| <= margin
    #
    # Linearisation around q0:
    #
    #   h(q) ≈ h0 + Jh0 @ (q - q0)
    #
    # and require:
    #
    #   -margin <= h(q) <= margin
    #
    # This produces 2 inequalities per equality constraint:
    #
    #   1. Jh0 @ q <= margin - h0 + Jh0 @ q0
    #   2. -Jh0 @ q <= margin + h0 - Jh0 @ q0
    # ------------------------------------------------------------------

    eps_h = margin * np.ones_like(h0)

    A_h = np.vstack([
        Jh0,
        -Jh0,
    ])

    b_h = np.concatenate([
        eps_h - h0 + Jh0 @ q0,
        eps_h + h0 - Jh0 @ q0,
    ])

    # ------------------------------------------------------------------
    # 2. Linearise the grasp inequalities g(q) <= 0 + margin
    #
    #   g(q) ≈ g0 + Jg0 @ (q - q0) <= 0 + margin
    #
    # => Jg0 @ q <= margin - g0 + Jg0 @ q0
    # ------------------------------------------------------------------

    A_g = Jg0

    b_g = margin - g0 + Jg0 @ q0

    # ------------------------------------------------------------------
    # 3. Add joint limits exactly
    #
    #   lower <= q <= upper
    # ------------------------------------------------------------------

    lower = np.asarray(grasp_constraints["lower"], dtype=float)
    upper = np.asarray(grasp_constraints["upper"], dtype=float)

    A_limits = np.vstack([
        np.eye(nq),
        -np.eye(nq),
    ])

    b_limits = np.concatenate([
        upper,
        -lower,
    ])

    # ------------------------------------------------------------------
    # 4. Combine everything
    #
    # Final set:
    #
    #   P = {q in R^nq | A q <= b}
    # ------------------------------------------------------------------

    A = np.vstack([
        A_h,
        A_g,
        A_limits,
    ])

    b = np.concatenate([
        b_h,
        b_g,
        b_limits,
    ])

    return HPolyhedron(A, b)


grasp_polytopes = [linearised_grasp_polytope(gc, q0_k) for gc, q0_k in zip(cap_grasp_constraints, q0s)]

# ------------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------------

for k, (G, gc, q0_k) in enumerate(zip(grasp_polytopes, cap_grasp_constraints, q0s)):
    print(f"cap {k + 1}: A {G.A().shape}, q0 inside: {G.PointInSet(q0_k)}")
    print(f"  h(q0): {gc['h'](q0_k)}")
    print(f"  g(q0): {gc['g'](q0_k)}")

cap 1: A (33, 11), q0 inside: True
  h(q0): [-1.00000118e-06 -9.99998824e-07  1.00000083e-06 -9.99998883e-07]
  g(q0): [-0.00033578 -0.00466422 -0.1       ]
cap 2: A (33, 11), q0 inside: True
  h(q0): [ 9.99995289e-07 -9.99996875e-07  9.99997537e-07 -9.99987588e-07]
  g(q0): [-1.00924824e-12 -5.00000000e-03 -1.00000000e-01]


In [46]:
placement_lower = plant.GetPositionLowerLimits().copy()
placement_upper = plant.GetPositionUpperLimits().copy()
placement_lower[idx_wrist], placement_upper[idx_wrist] = -1.0, 1.0
placement_lower[idx_f1], placement_upper[idx_f1] = -0.055, -0.024
placement_lower[idx_f2], placement_upper[idx_f2] = 0.024, 0.055
placement_polytope = HPolyhedron.MakeBox(placement_lower, placement_upper)

print(f"grasp polytopes: ambient_dimension={[G.ambient_dimension() for G in grasp_polytopes]}")
print(f"placement_polytope: ambient_dimension={placement_polytope.ambient_dimension()}")
print("every q0 in placement:", all(placement_polytope.PointInSet(q0_k) for q0_k in q0s))

grasp polytopes: ambient_dimension=[11, 11]
placement_polytope: ambient_dimension=11
every q0 in placement: True


## 3. Manipulation Planner class

Copied verbatim from `algorithms/manipulation_planner/manipulation_planner.py`
(unmodified - same pattern as `manipulation_planner_3dof.ipynb`/
`manipulation_planner_4dof.ipynb`, which inline the class in a cell rather
than importing it). Its `_generate_cfree` is the "IRIS-ZO original Drake
implementation" step: `IrisZoOptions` + `IrisInConfigurationSpaceFromCliqueCover`
straight from `pydrake`, not this project's Python reimplementation from
#69-71.

References `visualizer`, `panda_arm`, `panda_hand`, `cap` as plain globals
(inside `display_trajectory`) - already defined by section 1, so no
module-global wiring is needed here the way the imported version needs
(compare `full_arm_nhr.ipynb`'s `_manipulation_planner_module.visualizer = visualizer` cell).

In this notebook only its `_generate_cfree` is used (section 4); the A* in section 5 replaces the MIQP.

In [47]:
def convex_hull_to_hpolyhedron(hull):
    A = hull.equations[:, :-1]
    b = -hull.equations[:, -1]
    return HPolyhedron(A, b)


class ManipulationPlanner():

    def __init__(self,
            visualizer: CIrisPlantVisualizer,
            CP: HPolyhedron,
            CG: HPolyhedron,
            max_grasps: int = 20,
            model_instances: list = None,
            gripper_dim: int = None,
            cs_free: list[HPolyhedron] = None,
            irisalg: str = "irisnp"
        ):
        self.builder = visualizer.builder
        self.plant = visualizer.plant
        self.plant_context = visualizer.plant_context
        self.scene_graph = visualizer.scene_graph
        self.q_star = visualizer.q_star
        self.rat_fk = visualizer.rat_forward_kin
        self.inspector = visualizer.model_inspector
        self.diagram = visualizer.task_space_diagram
        self.diagram_context = visualizer.task_space_diagram_context
        self.lower_joint_limits = visualizer.q_lower_limits
        self.upper_joint_limits = visualizer.q_upper_limits
        self.visualize_cspace = visualizer.visualize_collision_constraint
        self.cspace_free_polytope = visualizer.cspace_free_polytope

        self.CP = CP
        self.CG = CG
        self.max_grasps = max_grasps

        self.model_instances = model_instances if model_instances is not None else [self.plant.GetModelInstanceByName("robot")]

        self.q_dim = self.plant.num_positions()
        self.gripper_dim = gripper_dim if gripper_dim is not None else self.q_dim


        self.cs_free = cs_free if cs_free is not None else self._generate_cfree(irisalg=irisalg)

    def set_max_grasps(self, max_grasps: int):
        self.max_grasps = max_grasps


    def compute_trajectory(self, x_init, x_goal, display=True):
        print("Finding path for minimum grasps")
        path = self._find_minimum_grasp_path(
                x_init = x_init,
                x_goal = x_goal,
                P = self.CP,
                G = self.CG,
                lower_joint_limits = self.lower_joint_limits,
                upper_joint_limits = self.upper_joint_limits,
                c_free_polytopes = self.cs_free
            )

        print("Path found. Generating trajectory")
        traj = self._generate_trajectory(path)


        print("Trajectory generated. Starting display")
        if display:
            self.display_trajectory(traj)

        return path, traj

    def display_trajectory(self, traj, meshcat=True, plotly=False):
        if meshcat:
            num_points = int((traj.end_time() - traj.start_time()) * 4000)
            for t in np.linspace(traj.start_time(), traj.end_time(), num_points):
                q = traj.value(t)
                # self.plant.GetJointByName("left_finger_sliding_joint", gripper).set_translation(self.plant_context, q[2])
                # # self.plant.GetJointByName("right_finger_sliding_joint", gripper).set_translation(self.plant_context, q[3])
                # self.plant.GetJointByName("base_revolute_joint", gripper).set_angle(self.plant_context, q[1])
                # self.plant.GetJointByName("cap_to_base", cap).set_angle(self.plant_context, -q[0])
                self.plant.GetJointByName("panda_finger_joint1", panda_hand).set_translation(self.plant_context, q[1][0])
                self.plant.GetJointByName("panda_joint7", panda_arm).set_angle(self.plant_context, q[0][0])
                self.plant.GetJointByName("cap_to_base", cap).set_angle(self.plant_context, -q[2][0])
                self.diagram.ForcedPublish(self.diagram_context)
                time.sleep(0.01)
        path = [traj.value(t) for t in np.linspace(traj.start_time(), traj.end_time(), 100)]
        path_q = [q.ravel() for q in path] # Convert to list of numpy arrays
        if plotly:
            self.visualize_cspace(factor=1, num_points=30, paths=[path_q], filled_polytopes=self.cs_free)
        return path_q

    def _generate_trajectory(self, path):
        trajs = []
        print("Generating trajectory from:")
        for q0, q1 in zip(path[:-1], path[1:]):
            print("From\t{}\tto\t{}".format(q0, q1))
            traj, result = self._generate_edge_trajectory(q0, q1)
            if not result.is_success():
                print("failed to generate edge trajectory from {} to {}".format(q0, q1))
                return None
            trajs.append(traj)
        return CompositeTrajectory.AlignAndConcatenate(trajs)


    def _generate_edge_trajectory(self, x_init, x_goal, path_length_weight=None):
        trajopt = GcsTrajectoryOptimization(self.plant.num_positions())
        gcs_regions = trajopt.AddRegions(self.cs_free, order=1, h_min=0.01)
        source = trajopt.AddRegions([Point(x_init)], order=0)
        target = trajopt.AddRegions([Point(x_goal)], order=0)
        trajopt.AddEdges(source, gcs_regions)
        trajopt.AddEdges(gcs_regions, target)
        trajopt.AddPathLengthCost(path_length_weight if path_length_weight is not None else 1.0)
        options = GraphOfConvexSetsOptions()
        [traj, result] = trajopt.SolvePath(source, target, options)
        print(f"result.is_success() = {result.is_success()}")
        print(f"result.get_solution_result() = {result.get_solution_result()}")
        print(f"result.get_solver_id().name() = {result.get_solver_id().name()}")


        # print("\n--- GCS RESULT ---")
        # print("success:", result.is_success())
        # print("solution result:", result.get_solution_result())
        # print("solver:", result.get_solver_id().name())

        # # try:
        # #     details = result.get_solver_details()
        # #     print("solver details:", details)

        # #     for field in [
        # #         "rescode",
        # #         "solution_status",
        # #         "optimizer_time",
        # #     ]:
        # #         if hasattr(details, field):
        # #             print(f"{field}:", getattr(details, field))
        # # except Exception as e:
        # #     print("Could not read solver details:", e)

        # print("------------------\n")

        return traj, result


    def _find_minimum_grasp_path(
            self,
            x_init: np.ndarray,
            x_goal: np.ndarray,
            P: HPolyhedron,
            G: HPolyhedron,
            lower_joint_limits: np.ndarray,
            upper_joint_limits: np.ndarray,
            c_free_polytopes: list[HPolyhedron]
            ) -> np.ndarray:

        for j in range(self.max_grasps):
            i = j + 1
            if i % 10 == 0 and i > 0:
                print(f"Trying with {i} grasps")
            path = self._solve_for_n_grasps_CC(
                i,
                x_init,
                x_goal,
                P,
                G,
                lower_joint_limits,
                upper_joint_limits,
                c_free_polytopes
            )

            if path is not None:
                print(f"Found a solution with {i} grasps")
                return path
        print(f"No solution found for less than {self.max_grasps} grasps")


    def _solve_for_n_grasps_CC(
            self,
            n_grasps: int,
            x_init: np.ndarray,
            x_goal: np.ndarray,
            P: HPolyhedron,
            G: HPolyhedron,
            lower_joint_limits: np.ndarray,
            upper_joint_limits: np.ndarray,
            c_free_polytopes: list[HPolyhedron]
            ) -> np.ndarray:
        # Initialize the program
        prog = MathematicalProgram()
        n_points = 2 * n_grasps + 2 # each grasp has a grasp and a release point, plus the initial and goal points
        n_vars = self.q_dim * n_points # for each point of q_dim length, we have n_points*q_dim variables
        non_gripper_dim = self.q_dim - self.gripper_dim # number of non-gripper variables per point

        # Create decision variables
        x = prog.NewContinuousVariables(n_vars, "x")

        # 1. Cost function: Minimize sum of squared distances between consecutive points
        """ ||x_{i+1} - x_{i}||^2  equiv || Cx - c ||^2  == 0 """

        c = np.zeros((n_vars - self.q_dim, 1))
        C = np.zeros((n_vars - self.q_dim, n_vars))
        C[:, self.q_dim:] = np.eye(n_vars - self.q_dim)
        C[:, :-self.q_dim] -= np.eye(n_vars - self.q_dim)

        Q, b = self._to_quadratic_form(C, c)
        Q += np.eye(Q.shape[1]) * 1e-6  # Add a small value to the diagonal to make Q positive definite

        prog.AddQuadraticCost(Q=Q, b=b, vars=x)

        # 2. Initial and goal constraints
        prog.AddLinearEqualityConstraint(np.eye(self.q_dim), x_init.flatten(), x[:self.q_dim])
        prog.AddLinearEqualityConstraint(np.eye(self.q_dim), x_goal.flatten(), x[-self.q_dim:])


        ###### Placement and Grasping Constraints should be adapted task specific #####

        # 3. Placement constraints (equality) -> Cap remains constant in transit paths
        """x_cap_{i+1} - x_cap_{i} == 0 for all even i, odd i+1"""
        for i in range(n_grasps + 1):
            idx = 2*self.q_dim*i # points to first variable of x_i for even i's
            idx1_cap = idx + self.q_dim - 1 # points to last variable of x_i (cap orientation)
            idx2_cap = idx1_cap + self.q_dim # points to last variable of x_{i+1} (cap orientation)
            prog.AddLinearEqualityConstraint(x[idx2_cap] - x[idx1_cap] == 0)

        # 4. Grasp constraints (equality between gripper and cap orientations)
        """x_cap_i - x_wrist_i - x_cap_{i+1} + x_wrist_{i+1} == 0 for odd i, even i+1"""
        for i in range(n_grasps):

            idxp1 = 2*self.q_dim*(i+1) # this idx points to first variable of x_i+1
            idx   = idxp1 - self.q_dim # this idx points to first variable of x_i

            idxp1_wrist = idxp1 + non_gripper_dim
            idxp1_cap = idxp1 + self.q_dim - 1
            idx_wrist = idx + non_gripper_dim
            idx_cap = idx + self.q_dim - 1 # equiv to idxp1 - 1

            prog.AddLinearEqualityConstraint(x[idx_cap] - x[idx_wrist] - x[idxp1_cap] + x[idxp1_wrist] == 0)

        ###### end of manual constraints


        # 5. Inequality Constraints (Placement hull)
        for i in range(n_points):
            prog.AddLinearConstraint(
                P.A(),  # Coefficient matrix
                -np.inf * np.ones_like(P.b()),  # Lower bound
                P.b(),  # Upper bound
                x[self.q_dim*i:self.q_dim*(i+1)]
            )
        # 6. Inequality Constraints (Grasp hull)
        # for i in range(n_points - 2):
        #     prog.AddLinearConstraint(
        #         G.A(),
        #         -np.inf * np.ones_like(G.b()),
        #         G.b(),
        #         x[self.q_dim*(i+1):self.q_dim*(i+2)]
        #     )

        # 7. Joint limits (inequality)
        prog.AddBoundingBoxConstraint(
            np.tile(lower_joint_limits, n_points),
            np.tile(upper_joint_limits, n_points),
            x
        )

        # 7. Collision-free polytope constraints (MIP)
        # Ensure all points are contained in at least one polytope

        M = 1e6  # Big-M constant (adjust based on problem scale)
        for i in range(n_points):
            x_i = x[self.q_dim*i:self.q_dim*(i+1)]

            # Creates one bineary 0-1 variable for each polytope
            z_i = prog.NewBinaryVariables(len(c_free_polytopes), name=f"z_{i}")

            # Each point x_i must be contained in at least one polytope
            prog.AddLinearConstraint(sum(z_i) >= 1)

            for j, poly in enumerate(c_free_polytopes):
                A_j = poly.A()
                b_j = poly.b()
                z_j = z_i[j]

                # Add constraints for each row of the polytope
                for k in range(A_j.shape[0]):
                    # Construct coefficient matrix [A_j_row | M]
                    coeffs = np.hstack([A_j[k], M])
                    # Combine x_i and z_j into variable vector
                    vars = np.hstack([x_i, [z_j]])
                    # A_j @ x_i + M * z[j] <= b_j + M
                    prog.AddLinearConstraint(
                        coeffs,
                        -np.inf,
                        b_j[k] + M,
                        vars
                    )

        # 8. Connected components constraints
        connected_components = self._compute_connected_components(P, G, c_free_polytopes)
        K = len(connected_components)

        # create binary variables for component membership (excluding init ang goal)
        z = {}
        for i in range (1, n_points-1): # exclude init and goal
            z[i] = prog.NewBinaryVariables(K, f"z_{i}")
            # Each point must belong to exactly one component
            prog.AddLinearConstraint(sum(z[i]) == 1)

        # Consecutive points must belong to the same component, ignoring init and goal
        for i in range(1, n_points-2, 2):
            for k in range(K):
                prog.AddLinearConstraint(z[i][k] == z[i+1][k])

        # Enforce that each component contains at least one point
        for i in range(1, n_points-1):
            x_i = x[self.q_dim*i:self.q_dim*(i+1)]
            z_i = z[i]

            for k, comp in enumerate(connected_components):
                A_k = comp.A()
                b_k = comp.b()
                z_k = z_i[k]

                for j in range(A_k.shape[0]):
                    coeffs = np.hstack([A_k[j], M])
                    vars = np.hstack([x_i, [z_k]])
                    prog.AddLinearConstraint(coeffs, -np.inf, b_k[j] + M, vars)

        # Solve the problem
        solver = MosekSolver()
        result = solver.Solve(prog)

        if not result.is_success():
            return None

        return result.GetSolution(x).reshape(-1, self.q_dim)

    @staticmethod
    def _to_quadratic_form(C, c):
        return np.dot(C.T, C), -np.dot(C.T, c).flatten()

    def _compute_connected_components(
        self,
        placement: HPolyhedron,
        grasp: HPolyhedron,
        c_free: list[HPolyhedron]
    ) -> list[HPolyhedron]:
        """
        Computes convex connected components as:
        {placement ∩ grasp ∩ (union of c_free)}.
        Returns convex hulls of connected regions.
        """
        components = []

        # Compute intersections with each c_free polytope
        for poly in c_free:
            intersection = placement.Intersection(grasp).Intersection(poly)
            if not intersection.IsEmpty():
                components.append(intersection)

        # Merge overlapping components
        merged = []
        for comp in components:
            add_to_merged = False
            for m in merged:
                if comp.Intersection(m).IsEmpty():
                    continue
                # Merge overlapping components
                vertices_comp = visualizer.get_polytope_vertices(comp)
                vertices_m = visualizer.get_polytope_vertices(m)
                vertices = np.vstack([vertices_comp, vertices_m])
                # QJ (joggle input): scipy's own recommendation for ConvexHull
                # precision failures in 5-D+ - near-degenerate/thin regions
                # (e.g. a very anisotropic grasp polytope) can otherwise raise
                # QhullError here.
                merged_hull = ConvexHull(vertices, qhull_options='QJ')
                
                vertices = np.vstack([vertices_comp, vertices_m])

                # Remove exactly/nearly duplicated vertices
                vertices = np.unique(np.round(vertices, decimals=12), axis=0)

                centered = vertices - vertices.mean(axis=0)

                s = np.linalg.svd(centered, compute_uv=False)

                print("Number of vertices:", len(vertices))
                print("Affine rank:", np.linalg.matrix_rank(centered))
                print("Singular values:")
                print(s)
                print("relative singular values:")
                print(s / s[0])


                merged_poly = convex_hull_to_hpolyhedron(merged_hull)
                merged.remove(m)
                merged.append(merged_poly)
                add_to_merged = True
                break
            if not add_to_merged:
                merged.append(comp)

        return merged



    def _generate_cfree(self, irisalg: str = "irisnp") -> list[HPolyhedron]:
        if irisalg not in ["irisnp", "irisnp2", "iriszo"]:
            raise ValueError(f"Invalid irisalg: {irisalg}. Must be one of ['irisnp', 'irisnp2', 'iriszo']")

        generator = RandomGenerator(1234)
        checker = SceneGraphCollisionChecker(
            model=self.diagram,
            robot_model_instances=self.model_instances,
            edge_step_size=0.01,
        )


        if irisalg != "irisnp":
            # For iriszo and irisnp2, we need to set the sampled iris options
            common_sampled_iris_options = CommonSampledIrisOptions()
            common_sampled_iris_options.delta = 0.001
            common_sampled_iris_options.epsilon = 0.001
            common_sampled_iris_options.max_iterations = 50
            common_sampled_iris_options.max_iterations_separating_planes = 50
            # common_sampled_iris_options.mixing_steps =
            common_sampled_iris_options.num_particles = 500
            common_sampled_iris_options.parallelism = Parallelism(32)
            common_sampled_iris_options.verbose = True
            common_sampled_iris_options.configuration_space_margin = 1e-4
            common_sampled_iris_options.termination_threshold = 1e-4
            common_sampled_iris_options.relative_termination_threshold = 1e-4
            common_sampled_iris_options.remove_all_collisions_possible = False

        options = IrisFromCliqueCoverOptions()
        options.num_points_per_visibility_round = 200
        options.coverage_termination_threshold = 0.98
        options.parallelism = Parallelism(32)

        if irisalg == "iriszo":
            iris_zo_options = IrisZoOptions()
            iris_zo_options.bisection_steps = 50
            iris_zo_options.sampled_iris_options = common_sampled_iris_options
            options.iris_options = iris_zo_options
        elif irisalg == "irisnp2":
            iris_np2_options = IrisNp2Options()
            iris_np2_options.sampled_iris_options = common_sampled_iris_options
            options.iris_options = iris_np2_options
        else:
            # irisnp uses the default IrisOptions object on options.iris_options.
            options.iris_options.configuration_space_margin = 1e-4

        # See https://github.com/RobotLocomotion/drake/issues/21343
        regions = IrisInConfigurationSpaceFromCliqueCover(checker, options, generator, [])
        print("\nnumber of regions: ", len(regions))
        return regions

## 4. Generate C-free for this scene (native Drake IRIS-NP2 clique cover)

The single-cap notebook's C-free files are 10-D and include its obstacles, so they do not fit here.
`_generate_cfree` covers the full nq-D configuration space (all caps included; 99% coverage,
`Parallelism(32)`), which takes a while. The result is cached in `data/cfree/` under a name that records
the number of DOF, caps and obstacles, and later runs load it from there. **Delete the file to regenerate
after moving caps or obstacles**: the name does not record their poses.

In [48]:
from pydrake.geometry.optimization import LoadIrisRegionsYamlFile, SaveIrisRegionsYamlFile

model_instances = [panda_arm, panda_hand, *caps, *obstacles]
cfree_file = REPO_ROOT / "data" / "cfree" / f"cfree_{nq}dof_{K}caps_{len(obstacles)}obs_irisnp2_99.yaml"
cfree_file = REPO_ROOT / "data" / "cfree" / f"cfree_11dof_2caps_0obs_irisnp2_98_capsafe.yaml"
cfree_regions = None
if cfree_file.exists():
    cfree_regions = list(LoadIrisRegionsYamlFile(str(cfree_file)).values())
    print(f"loaded {len(cfree_regions)} regions from {cfree_file.name}")

t_start = time.time()
planner = ManipulationPlanner(
    visualizer=visualizer,
    CP=placement_polytope,
    CG=grasp_polytopes[0],      # unused by the A* below, which has one grasp polytope per cap
    max_grasps=20,
    gripper_dim=3,
    model_instances=model_instances,
    cs_free=cfree_regions,      # None -> generate
    irisalg="irisnp2"
)

if cfree_regions is None:
    print(f"generated {len(planner.cs_free)} regions in {time.time() - t_start:.1f}s")
    SaveIrisRegionsYamlFile(
        str(cfree_file),
        {f"region_{i:03d}": region for i, region in enumerate(planner.cs_free)},
    )
    print(f"saved to {cfree_file}")

loaded 33 regions from cfree_11dof_2caps_0obs_irisnp2_98_capsafe.yaml


## Fixing free polytopes

**Only runs when the scene has obstacles** (`OBSTACLE_POSES`). Without them, the IRIS polytopes are used
as they are. With several caps, the sweep below is done from each cap's grasp seed in turn, and the NLP
keeps every cap angle fixed.

Some of the free polytopes generated by either iris-zo and iris-np2 collide with the obstacles in some of the polytopes. Here, we'll try to go free polytope by free polytope that intersects with the grasp polytope. In the intersection with the grasp polytope, we interpolate along the wrist axis and find collisions with the collision checker. We then add hyperplanes to the free polytope accordingly to remove collisions, in perpendicular direction to the wrist axis.

1. **Wrist sweep.** Same sweep as the brute force below: `grasp_configuration` with the wrist from −1 to 1, each sample checked with the collision checker.
2. **Free segment.** For every free polytope F, the collision-free samples inside it form its free wrist segment. We keep the longest one, as the brute force does.
3. **False positives.** A sample the collision checker says collides but some F says is free. For every false positive q_c and every F that contains it, add one plane to F:
   - NLP (only constraint: don't collide): q_f = argmin ‖q − q_c‖², seeded at q_c, cap fixed. q_f is on the border of the obstacle and q_c − q_f is the direction that gets out of collision, so the plane (q_c − q_f)ᵀ q ≤ (q_c − q_f)ᵀ q_f goes through q_f, is tangent to the obstacle there and cuts q_c off.
   - If the NLP fails or its plane would also cut the free segment of F, use a plane perpendicular to the wrist axis at the end of the free segment instead.
4. **Check.** Count the false positives again and replace `planner.cs_free` with the fixed polytopes, so the brute force and the A* below use them.

**Notes.**
- The fingers must stay at ±0.0245 as in the brute force. At q0's ±0.02405 the finger spheres already touch the cap (~0.1 mm), which the collision checker counts as a collision at every wrist angle.
- The cap angle does not change the collisions (cap 0.94 and 3.0 give the same result), so dropping the cap in the A* is fine.
- The NLP gets out of collision mostly by moving the other arm joints by ~0.01 rad (the obstacle penetration is only 4-6 mm), not the wrist. Its plane is then almost parallel to the wrist axis and would also cut the free segment, so in practice the planes added are the wrist ones.

In [49]:
NLP_MARGIN = 1e-3    # distance the NLP asks for, a bit more than the collision checker's 1e-5

e_wrist = np.zeros(plant.num_positions())
e_wrist[idx_wrist] = 1.0


def longest_free_run(sweep_in_F):
    """Longest run of consecutive collision-free samples: the free wrist segment of a polytope (as in the brute force)."""
    best, run = [], []
    for q, free in sweep_in_F:
        run = run + [q] if free else []
        if len(run) > len(best):
            best = run
    return best


def separating_plane(q_c, keep):
    """Plane n @ q <= d that cuts the colliding configuration q_c off and keeps the samples in keep.

    NLP (only constraint: don't collide): q_f = argmin ||q - q_c||^2, seeded at q_c, cap fixed.
    q_f is on the border of the obstacle and n = q_c - q_f is the direction that gets out of
    collision, so the plane through q_f with normal n is tangent to the obstacle there.
    Fallback (NLP fails, or its plane would also cut keep): plane perpendicular to the wrist axis
    at the end of the free segment.
    """
    ik = InverseKinematics(plant, plant_context)
    ik.AddMinimumDistanceLowerBoundConstraint(NLP_MARGIN)
    q = ik.q()
    prog = ik.get_mutable_prog()
    for i_cap in idx_caps:                           # no cap component in the plane
        prog.AddBoundingBoxConstraint(q_c[i_cap], q_c[i_cap], q[i_cap])
    prog.AddQuadraticErrorCost(np.eye(len(q_c)), q_c, q)
    result = Solve(prog, q_c)
    if result.is_success():
        q_f = result.GetSolution(q)
        n = q_c - q_f
        if np.linalg.norm(n) > 1e-9:
            n = n / np.linalg.norm(n)
            d = n @ q_f
            if all(n @ q_k <= d + 1e-9 for q_k in keep):
                return n, d, "nlp"
    if not keep:
        return None
    w_keep = [q_k[idx_wrist] for q_k in keep]
    if q_c[idx_wrist] > max(w_keep):
        return e_wrist, max(w_keep), "wrist"      # wrist <= end of the free segment
    return -e_wrist, -min(w_keep), "wrist"        # wrist >= start of the free segment

In [50]:
if not hasattr(planner, "cs_free_iris"):   # keep the IRIS polytopes, so this cell can be re-run
    planner.cs_free_iris = list(planner.cs_free)
cfree_fixed = list(planner.cs_free_iris)
wrist_angles = np.linspace(-1., 1., num=100)

sweeps = []   # one wrist sweep per cap: (q, collision-free according to the collision checker)
if not obstacles:
    print("no obstacles: nothing to fix, the IRIS polytopes are used as they are")

for cap_k, gc in enumerate(cap_grasp_constraints if obstacles else []):
    # Same wrist sweep as the single-cap brute force, from this cap's grasp seed
    grasp_configuration = find_grasp_seed(gc)
    grasp_configuration[idx_f1] = -0.0245
    grasp_configuration[idx_f2] = 0.0245
    grasp_configuration[idx_caps] = 3.0
    sweep = []
    for w in wrist_angles:
        q = grasp_configuration.copy()
        q[idx_wrist] = w
        sweep.append((q, not visualizer.check_collision_q_by_ik(q)))
    sweeps.append(sweep)

    # free wrist segment of every polytope: longest run of free samples inside it
    keep = [longest_free_run([(q, free) for q, free in sweep if F.PointInSet(q)]) for F in cfree_fixed]

    planes = {}   # k -> kinds of the planes added to F_k
    for q, free in sweep:
        if free:
            continue
        for k, F in enumerate(cfree_fixed):
            if not F.PointInSet(q):
                continue
            # false positive: the collision checker says collision, F_k says free
            plane = separating_plane(q, keep[k])
            if plane is None:
                print(f"cap {cap_k + 1}: F_{k}: no plane for the false positive at wrist {q[idx_wrist]:+.3f}")
                continue
            n, d, kind = plane
            cfree_fixed[k] = HPolyhedron(np.vstack([F.A(), n]), np.append(F.b(), d))
            planes.setdefault(k, []).append(kind)

    for k, kinds in planes.items():
        seg = f"[{keep[k][0][idx_wrist]:+.3f}, {keep[k][-1][idx_wrist]:+.3f}]" if keep[k] else "none"
        print(f"cap {cap_k + 1}: F_{k:<3d} free segment {seg}  planes: {kinds.count('nlp')} nlp, {kinds.count('wrist')} wrist")

no obstacles: nothing to fix, the IRIS polytopes are used as they are


In [51]:
def count_false_positives(polytopes):
    return sum(not free and any(F.PointInSet(q) for F in polytopes) for sweep in sweeps for q, free in sweep)


if sweeps:
    print(f"false positives: {count_false_positives(planner.cs_free_iris)} before, "
          f"{count_false_positives(cfree_fixed)} after (out of {sum(len(s) for s in sweeps)} wrist angles)")

planner.cs_free = cfree_fixed   # the A* below uses the fixed polytopes

## Reachability A* over free and grasp polytopes

Inspired by CASSR (Wang & Tonneau, arXiv 2603.02989): an A* **node is a polytope plus the subset of it
reachable from the start**, not a single configuration. Expanding a node propagates reachability into
neighbouring polytopes.

In our problem the state is (arm configuration q, cap angles α = (α₁, …, α_K)):
- **Transit** in a free region F_i: the arm can reach any point of F_i (it is convex), and every α is frozen.
- **Grasp** in Gₖ = Grasp_k ∩ F (cap k's grasp polytope): the wrist is the only real degree of freedom.
  A forward stroke (wrist ↑) turns cap k, αₖ −= Δw. A return stroke (wrist ↓) keeps αₖ (ratchet). The
  other caps are not touched.

So a node's reachable set is always *(the whole polytope) × (one interval of α per cap)*, and every move
changes at most one cap's interval: propagation is still interval arithmetic, cap by cap. A* minimises the
number of grasps (forward strokes), ties broken by fewer polytope hops. The actual configurations are
placed afterwards, on the chosen polytope sequence.

### Step 1 — Drop the caps from every polytope
We ignore the caps in the polytopes: their columns are deleted from every C-free region and from every
grasp polytope (rows that only bounded a cap, i.e. its joint limits, disappear). The cap angles α are
taken from q_init / q_goal and only change through the grasp pieces (steps 5-6).

**Trust region on each grasp polytope.** Each grasp polytope linearises h(q) around its q0ₖ, but Jh only
has rank 4, so the linear constraints leave a 6-D null space free: the arm could slide ~2 rad away from
q0ₖ (gripper 0.8 m from the cap, |h| ≈ 0.95) and still be "inside". The linearisation is only valid near
q0ₖ, so we keep the 6 non-wrist arm joints within ±`GRASP_TRUST` of q0ₖ and leave the wrist free: its axis
passes through the EE origin, so rotating it keeps the grasp. This also keeps the caps' grasp polytopes
apart (checked below).

In [52]:
import itertools
from pydrake.solvers import MathematicalProgram, Solve

cap_lower = plant.GetPositionLowerLimits()[idx_caps]
cap_upper = plant.GetPositionUpperLimits()[idx_caps]
arm_idx = [i for i in range(nq) if i not in idx_caps]   # the arm + finger coordinates
W = arm_idx.index(idx_wrist)                            # wrist index in the cap-free space


def to_arm(q):
    """nq-D configuration -> cap-free configuration (caps removed)."""
    return np.asarray(q, dtype=float)[arm_idx]


def to_full(x, alpha):
    """Cap-free configuration + cap angles alpha (one per cap) -> nq-D configuration."""
    q = np.empty(nq)
    q[arm_idx] = x
    q[idx_caps] = alpha
    return q


def drop_cap(poly, tol=1e-9):
    """Remove the cap coordinates: the caps are ignored in the polytopes and tracked separately (α)."""
    A_arm = np.delete(poly.A(), idx_caps, axis=1)
    keep = np.linalg.norm(A_arm, axis=1) > tol    # rows that only bounded a cap (its joint limits) vanish
    return HPolyhedron(A_arm[keep], poly.b()[keep])


free_sets = [drop_cap(F) for F in planner.cs_free]

# Trust region: the linearisation of h(q) is only valid near q0 -> keep the non-wrist arm joints near q0.
GRASP_TRUST = 0.05   # rad


def trust_region(q0_k):
    x0 = to_arm(q0_k)
    trust_lo, trust_hi = np.full(len(x0), -np.inf), np.full(len(x0), np.inf)
    for j in range(7):                                   # the 7 arm joints (0..6 in the plant ordering)
        if j == idx_wrist:
            continue
        jj = arm_idx.index(j)
        trust_lo[jj], trust_hi[jj] = x0[jj] - GRASP_TRUST, x0[jj] + GRASP_TRUST
    A_trust = np.vstack([np.eye(len(x0)), -np.eye(len(x0))])
    b_trust = np.concatenate([trust_hi, -trust_lo])
    finite = np.isfinite(b_trust)
    return HPolyhedron(A_trust[finite], b_trust[finite])


grasp9 = [drop_cap(G).Intersection(trust_region(q0_k)) for G, q0_k in zip(grasp_polytopes, q0s)]   # one per cap
print(f"grasp polytopes: trust region ±{GRASP_TRUST} rad on the non-wrist arm joints")
print(f"{len(free_sets)} free regions, {sum(F.IsEmpty() for F in free_sets)} empty after dropping the caps")
for k in range(K):
    print(f"cap {k + 1}: q0 (arm) in its grasp polytope:", grasp9[k].PointInSet(to_arm(q0s[k])))
for a, b in itertools.combinations(range(K), 2):
    print(f"grasp polytopes of caps {a + 1} and {b + 1} disjoint:", grasp9[a].Intersection(grasp9[b]).IsEmpty())

grasp polytopes: trust region ±0.05 rad on the non-wrist arm joints
33 free regions, 0 empty after dropping the caps
cap 1: q0 (arm) in its grasp polytope: True
cap 2: q0 (arm) in its grasp polytope: True
grasp polytopes of caps 1 and 2 disjoint: True


### Step 2 — Grasp pieces G_{k,i} = Grasp_k ∩ F_i
These are the switch sets: the only places where a cap can be turned, here cap k. Each piece is
essentially an interval of wrist values; two small LPs (no binaries, no big-M) find its lowest- and
highest-wrist points. Pieces are keyed by (cap, free region).

In [53]:
def extreme_point(poly, direction):
    """Point of poly minimising direction @ x (a small LP, no binaries)."""
    prog = MathematicalProgram()
    x = prog.NewContinuousVariables(poly.ambient_dimension(), "x")
    prog.AddLinearConstraint(poly.A(), np.full(len(poly.b()), -np.inf), poly.b(), x)
    prog.AddLinearCost(direction, 0.0, x)
    result = Solve(prog)
    if not result.is_success():
        raise RuntimeError("extreme_point LP failed")
    return result.GetSolution(x)


e_w = np.zeros(len(arm_idx))
e_w[W] = 1.0

pieces = {}   # (cap, i) -> piece of the cap's grasp polytope inside free region i
for cap_k in range(K):
    for i, F in enumerate(free_sets):
        G = grasp9[cap_k].Intersection(F)
        if G.IsEmpty():
            continue
        lo, hi = extreme_point(G, e_w), extreme_point(G, -e_w)
        pieces[(cap_k, i)] = dict(poly=G, lo=lo, hi=hi, w_lo=lo[W], w_hi=hi[W])

print(f"{len(pieces)} grasp pieces:")
for (cap_k, i), p in sorted(pieces.items(), key=lambda kv: (kv[0][0], kv[1]["w_lo"])):
    print(f"  cap {cap_k + 1}  G_{i:<3d} wrist [{p['w_lo']:+.3f}, {p['w_hi']:+.3f}]  length {p['w_hi'] - p['w_lo']:.3f}")

4 grasp pieces:
  cap 1  G_0   wrist [-1.000, -0.463]  length 0.537
  cap 1  G_1   wrist [-0.374, +1.000]  length 1.374
  cap 2  G_4   wrist [-1.000, +1.000]  length 2.000
  cap 2  G_16  wrist [-1.000, +1.000]  length 2.000


### Step 3 — Grasp components (wrist segments)
Pieces that intersect form one connected wrist segment of length L_c: this is what the brute force
found by sweeping, but derived from the polytopes. A forward stroke can turn the cap by at most L_c.
The chain is a wrist-ordered list of points, each consecutive pair inside one piece (so lerps are safe).

**`MERGE_PIECES = False` (for now):** every straight-line move in the final path must lie inside *one*
convex set, which is what certifies it collision-free. Inside a single piece that is automatic. Across
merged pieces, the entry piece may not be the one the stroke chain runs through, and that move would not
be covered by any single polytope. So each piece is its own grasp component for now. With the current
regions every piece already spans the whole wrist range, so nothing is lost.

With several caps, only pieces of the same cap are merged, and every component remembers its cap (`cap`),
the only one it can turn. `L_MAX` is the longest segment of each cap.

In [54]:
L_MIN = 0.05          # ignore segments too short to be useful for turning
MERGE_PIECES = False  # True: merge overlapping pieces into one segment (step 6 needs False)

piece_keys = sorted(pieces)
overlaps = {}
root = {k: k for k in piece_keys}


def find(k):
    while root[k] != k:
        root[k] = root[root[k]]
        k = root[k]
    return k


for a, b in itertools.combinations(piece_keys, 2):
    if a[0] != b[0]:
        continue                                # pieces of different caps never form one segment
    I = pieces[a]["poly"].Intersection(pieces[b]["poly"])
    if not I.IsEmpty():
        overlaps[(a, b)] = overlaps[(b, a)] = I
        root[find(a)] = find(b)

groups = {}
for k in piece_keys:
    groups.setdefault(find(k) if MERGE_PIECES else k, []).append(k)


def build_chain(member_keys):
    cur = min(member_keys, key=lambda k: pieces[k]["w_lo"])
    points, seg_piece = [pieces[cur]["lo"]], []
    while True:
        ahead = [k for k in member_keys
                 if (cur, k) in overlaps and pieces[k]["w_hi"] > pieces[cur]["w_hi"] + 1e-9]
        if not ahead:
            break
        nxt = max(ahead, key=lambda k: pieces[k]["w_hi"])
        points.append(extreme_point(overlaps[(cur, nxt)], -e_w))   # highest-wrist point of the overlap
        seg_piece.append(cur)
        cur = nxt
    points.append(pieces[cur]["hi"])
    seg_piece.append(cur)
    wrists = [p[W] for p in points]
    if np.any(np.diff(wrists) < -1e-9):
        print(f"  warning: chain not monotone in wrist: {np.round(wrists, 3)}")
    return points, seg_piece


components = []
for member_keys in groups.values():
    chain, seg_piece = build_chain(member_keys)
    L = chain[-1][W] - chain[0][W]
    if L < L_MIN:
        continue
    components.append(dict(cap=member_keys[0][0], pieces=member_keys, chain=chain, seg_piece=seg_piece,
                           w_min=chain[0][W], w_max=chain[-1][W], L=L))

L_MAX = np.zeros(K)   # longest segment of each cap
for c in components:
    L_MAX[c["cap"]] = max(L_MAX[c["cap"]], c["L"])
if np.any(L_MAX == 0):
    raise RuntimeError(f"no grasp component for cap(s) {[k + 1 for k in range(K) if L_MAX[k] == 0]}")

print(f"{len(components)} grasp components (segments), longest L per cap = {np.round(L_MAX, 3)}")
for c_id, c in enumerate(components):
    print(f"  C{c_id}: cap {c['cap'] + 1}  wrist [{c['w_min']:+.3f}, {c['w_max']:+.3f}]  L={c['L']:.3f}  "
          f"pieces {sorted(i for _, i in c['pieces'])}")

4 grasp components (segments), longest L per cap = [1.374 2.   ]
  C0: cap 1  wrist [-1.000, -0.463]  L=0.537  pieces [0]
  C1: cap 1  wrist [-0.374, +1.000]  L=1.374  pieces [1]
  C2: cap 2  wrist [-1.000, +1.000]  L=2.000  pieces [4]
  C3: cap 2  wrist [-1.000, +1.000]  L=2.000  pieces [16]


### Step 4 — Polytope graph
Edges connect sets with a nonempty intersection:
- F_i ↔ F_j (transit): door = Chebyshev centre of F_i ∩ F_j.
- F_i ↔ component c (switch): through a piece G_k with F_i ∩ G_k ≠ ∅. We store the highest wrist value
  there (best entry) and the lowest (best exit): moving the wrist forward while grasping always turns
  the cap, so exiting "ahead" of the entry forces some turning.

Computed eagerly here for clarity (CASSR builds this lazily, which is a later optimisation).

In [55]:
n_free = len(free_sets)

ff_door = {}      # (i, j) -> Chebyshev centre of F_i ∩ F_j
for i, j in itertools.combinations(range(n_free), 2):
    I = free_sets[i].Intersection(free_sets[j])
    if not I.IsEmpty():
        ff_door[(i, j)] = ff_door[(j, i)] = I.ChebyshevCenter()
free_neighbors = {i: [j for j in range(n_free) if (i, j) in ff_door] for i in range(n_free)}

fc_links = {}     # (i, c) -> list of dicts: piece k, entry/exit points and wrist values
for i in range(n_free):
    for c_id, comp in enumerate(components):
        links = []
        for k in comp["pieces"]:
            I = free_sets[i].Intersection(pieces[k]["poly"])
            if I.IsEmpty():
                continue
            entry, exit_ = extreme_point(I, -e_w), extreme_point(I, e_w)
            links.append(dict(piece=k, entry=entry, entry_w=entry[W], exit=exit_, exit_w=exit_[W]))
        if links:
            fc_links[(i, c_id)] = links
comps_near = {i: [c for c in range(len(components)) if (i, c) in fc_links] for i in range(n_free)}

print(f"transit edges F-F: {len(ff_door) // 2}")
print(f"switch edges F-C:  {len(fc_links)}")
for c_id in range(len(components)):
    print(f"  C{c_id} reachable from free regions {[i for i in range(n_free) if (i, c_id) in fc_links]}")

transit edges F-F: 456
switch edges F-C:  6
  C0 reachable from free regions [0]
  C1 reachable from free regions [1]
  C2 reachable from free regions [4, 16]
  C3 reachable from free regions [4, 16]


### Step 5 — A* search over (polytope, α-box) nodes

**Node** = a polytope plus the set reachable in it: `P × [α₁_lo, α₁_hi] × … × [α_K_lo, α_K_hi]`.
- `kind="F"`: transit inside free region F_i. Any arm configuration of F_i is reachable (convex), every α is frozen.
- `kind="C"`: inside grasp component c, which belongs to one cap k = `components[c]["cap"]`. Also remembers
  the wrist range where this visit entered (`entry_w_lo`, `entry_w`) and its **phase**:
  - `entered` — just arrived, nothing grasped yet (the wrist can be anywhere in the entry range);
  - `holding` — just finished a forward stroke, still grasping;
  - `released` — after a return (ungrasped, wrist rotated back), ready for a full stroke.

**Moves**, with the α rule and cost of each. Inside a component of cap k only αₖ's interval changes; every
other cap's interval passes through untouched. Cost = **grasp actions + ungrasp actions**, plus a small
price μ = `HOP_COST` per polytope change (approach motions, no contact):

| move | from → to | α intervals | cost |
|---|---|---|---|
| transit | F_i → F_j (F_i ∩ F_j ≠ ∅) | unchanged | μ |
| enter   | F_i → C (F_i ∩ G ≠ ∅) | unchanged | μ |
| stroke (grasp) | `entered` → `holding` | `αₖ_lo −= w_max − e_lo` (starts inside the entry range) | 1 |
| stroke (grasp) | `released` → `holding` | `αₖ_lo −= L` | 1 |
| return (ungrasp) | `entered`/`holding` → `released` | unchanged | 1 |
| exit (release) | `holding` → F_j | `αₖ_hi −= d` | 1 |
| exit (never grasped) | `entered` → F_j, only if d ≤ 0 | unchanged | μ |

d = max(0, lowest exit wrist − highest entry wrist): the forward wrist travel (= turning of cap k) that
reaching the exit forces. αₖ_lo is clipped at αₖ_goal.

Exit must cost 1: otherwise "exit and re-enter elsewhere" would be a free substitute for a return.
Note that a return *before* the first stroke is always avoidable with one extra hop: entering G_k from
F_i needs F_i ∩ G_k ≠ ∅, so F_i touches F_k, and F_k ∩ G_k = G_k covers the whole wrist range of the
piece. With μ < 1 the search therefore prefers that hop.

**Heuristic** h = Σₖ 2·⌈(αₖ_lo − αₖ_goal)/L_max,k⌉ (+1 if `holding`): every remaining stroke turns one cap
and must be followed by an ungrasp (a return or the final release), and a holding node still has to
release. Admissible, and consistent (no move lowers h by more than its cost) → A* returns the fewest
grasp + ungrasp actions, ties broken by fewer hops.

**Goal**: an F node whose region contains q_goal's arm configuration and whose interval contains αₖ_goal
for every cap.

**Pruning (exact dominance)**: skip a node if an already-expanded node with the same key (same polytope;
for C also the same entry range and phase) had a box containing this one (every cap's interval) at no
higher cost.

#### 5a — Node class, heuristic and dominance check

In [56]:
import heapq
from dataclasses import dataclass

HOP_COST = 1e-3   # μ: price of changing polytope (transit / enter). << 1 means "fewest actions first".
TOL = 1e-6


def with_k(t, k, value):
    """Tuple t with entry k replaced by value: a move changes one cap's bound, the others pass through."""
    return t[:k] + (float(value),) + t[k + 1:]


@dataclass
class ReachNode:
    kind: str                  # "F": transit in free region set_id, "C": in grasp component set_id
    set_id: int
    alpha_lo: tuple            # reachable angles of cap k in this node: [alpha_lo[k], alpha_hi[k]]
    alpha_hi: tuple
    g: float = 0.0             # cost so far: #grasps + #ungrasps + HOP_COST * #hops
    h: float = 0.0             # lower bound on the remaining #actions
    parent: "ReachNode" = None
    action: tuple = ("start",)
    entry_w: float = None      # C only: highest wrist value of (entry region ∩ G)
    entry_w_lo: float = None   # C only: lowest wrist value of (entry region ∩ G)
    phase: str = None          # C only: "entered", "holding" or "released"

    @property
    def f(self):
        return self.g + self.h

    def key(self):
        if self.kind == "F":
            return ("F", self.set_id)
        return ("C", self.set_id, round(self.entry_w, 6), round(self.entry_w_lo, 6), self.phase)

    def __repr__(self):
        name = f"F{self.set_id}" if self.kind == "F" else f"C{self.set_id}"
        phase = f" [{self.phase}]" if self.kind == "C" else ""
        alphas = "  ".join(f"α{k + 1}∈[{lo:+.3f}, {hi:+.3f}]"
                           for k, (lo, hi) in enumerate(zip(self.alpha_lo, self.alpha_hi)))
        return f"{name:<4} {alphas}  g={self.g:.3f}{phase}"


def actions_lower_bound(node, alpha_goal):
    n_strokes = 0
    for k in range(K):
        remaining = node.alpha_lo[k] - alpha_goal[k]
        if remaining > TOL:
            n_strokes += int(np.ceil((remaining - TOL) / L_MAX[k]))
    pending_release = 1.0 if node.kind == "C" and node.phase == "holding" else 0.0
    return 2.0 * n_strokes + pending_release


def dominated(node, expanded):
    return any(g <= node.g + TOL
               and all(lo[k] <= node.alpha_lo[k] + TOL and hi[k] >= node.alpha_hi[k] - TOL for k in range(K))
               for lo, hi, g in expanded.get(node.key(), []))


free_near_comp = {c: [i for i in range(n_free) if (i, c) in fc_links] for c in range(len(components))}

#### 5b — Successors (the four moves) and the A* loop
Each successor is `(child fields, edge cost, action)`. The action records which polytopes/pieces were
used, so step 6 can place the actual configurations on the chosen sequence.

In [57]:
def successors(node, alpha_goal):
    if node.kind == "F":
        i = node.set_id
        for j in free_neighbors[i]:                                       # transit
            yield (dict(kind="F", set_id=j, alpha_lo=node.alpha_lo, alpha_hi=node.alpha_hi),
                   HOP_COST, ("transit", i, j))
        for c in comps_near[i]:                                           # enter grasp component
            link = max(fc_links[(i, c)], key=lambda l: l["entry_w"])
            yield (dict(kind="C", set_id=c, alpha_lo=node.alpha_lo, alpha_hi=node.alpha_hi,
                        entry_w=link["entry_w"], entry_w_lo=link["exit_w"], phase="entered"),
                   HOP_COST, ("enter", i, c, link["piece"]))
        return

    c, comp = node.set_id, components[node.set_id]
    k = comp["cap"]                                                       # the only cap this component turns
    same = dict(kind="C", set_id=c, alpha_hi=node.alpha_hi, entry_w=node.entry_w, entry_w_lo=node.entry_w_lo)

    if node.phase in ("entered", "released"):                             # stroke = grasp
        reach = comp["w_max"] - node.entry_w_lo if node.phase == "entered" else comp["L"]
        new_lo = max(node.alpha_lo[k] - reach, alpha_goal[k], cap_lower[k])
        if new_lo < node.alpha_lo[k] - TOL:
            yield dict(same, alpha_lo=with_k(node.alpha_lo, k, new_lo), phase="holding"), 1.0, ("stroke", c)

    if node.phase in ("entered", "holding"):                              # return = ungrasp
        yield dict(same, alpha_lo=node.alpha_lo, phase="released"), 1.0, ("return", c)

        for j in free_near_comp[c]:                                       # exit
            link = min(fc_links[(j, c)], key=lambda l: l["exit_w"])
            d = max(0.0, link["exit_w"] - node.entry_w)                   # forced forward travel = forced turning
            if node.phase == "entered" and d > TOL:
                continue                                                  # reaching that exit needs a stroke
            new_hi = node.alpha_hi[k] - d
            if new_hi < node.alpha_lo[k] - TOL:
                continue
            cost = 1.0 if node.phase == "holding" else HOP_COST          # releasing the cap is an ungrasp
            yield (dict(kind="F", set_id=j, alpha_lo=node.alpha_lo, alpha_hi=with_k(node.alpha_hi, k, new_hi)),
                   cost, ("exit", c, j, link["piece"]))


def reachability_astar(q_init, q_goal, max_expansions=200_000):
    alpha_init = tuple(float(a) for a in np.asarray(q_init)[idx_caps])
    alpha_goal = tuple(float(a) for a in np.asarray(q_goal)[idx_caps])
    for k in range(K):
        if alpha_goal[k] > alpha_init[k] + TOL:
            raise ValueError(f"cap {k + 1}: α_goal={alpha_goal[k]:.3f} > α_init={alpha_init[k]:.3f}: "
                             f"the cap only turns one way (α decreases)")
        if alpha_goal[k] < cap_lower[k] - TOL:
            raise ValueError(f"cap {k + 1}: α_goal={alpha_goal[k]:.3f} is below the cap joint limit {cap_lower[k]:.3f}")

    x_init, x_goal = to_arm(q_init), to_arm(q_goal)
    starts = [i for i, F in enumerate(free_sets) if F.PointInSet(x_init)]
    goal_sets = {i for i, F in enumerate(free_sets) if F.PointInSet(x_goal)}
    if not starts:
        raise ValueError("q_init's arm configuration is in no (cap-dropped) free region")
    if not goal_sets:
        raise ValueError("q_goal's arm configuration is in no (cap-dropped) free region")

    counter = itertools.count()
    open_heap = []
    for i in starts:
        node = ReachNode("F", i, alpha_init, alpha_init)
        node.h = actions_lower_bound(node, alpha_goal)
        heapq.heappush(open_heap, (node.f, next(counter), node))

    expanded = {}      # node key -> list of (alpha_lo, alpha_hi, g) already expanded
    n_expanded = 0
    while open_heap and n_expanded < max_expansions:
        _, _, node = heapq.heappop(open_heap)
        if dominated(node, expanded):
            continue
        expanded.setdefault(node.key(), []).append((node.alpha_lo, node.alpha_hi, node.g))
        n_expanded += 1

        if (node.kind == "F" and node.set_id in goal_sets
                and all(node.alpha_lo[k] <= alpha_goal[k] + TOL and alpha_goal[k] <= node.alpha_hi[k] + TOL
                        for k in range(K))):
            path = []
            while node is not None:
                path.append(node)
                node = node.parent
            path.reverse()
            return path, dict(expanded=n_expanded, starts=starts, goal_sets=sorted(goal_sets))

        for fields, cost, action in successors(node, alpha_goal):
            child = ReachNode(**fields, g=node.g + cost, parent=node, action=action)
            child.h = actions_lower_bound(child, alpha_goal)
            if not dominated(child, expanded):
                heapq.heappush(open_heap, (child.f, next(counter), child))

    return None, dict(expanded=n_expanded, starts=starts, goal_sets=sorted(goal_sets))


def count_actions(path):
    strokes = sum(n.action[0] == "stroke" for n in path)
    returns = sum(n.action[0] == "return" for n in path)
    releases = sum(n.action[0] == "exit" and n.parent.phase == "holding" for n in path)
    hops = sum(n.action[0] in ("transit", "enter") for n in path)
    per_cap = [sum(n.action[0] == "stroke" and components[n.action[1]]["cap"] == k for n in path)
               for k in range(K)]
    return dict(grasps=strokes, returns=returns, releases=releases, hops=hops, grasps_per_cap=per_cap)

#### 5c — Run it on two caps
Random start and goal configurations in C-free (seeded), with the cap angles set from `ALPHA_INIT` and
`ALPHA_GOAL`. Set an entry of `ALPHA_GOAL` equal to `ALPHA_INIT` to leave that cap alone.

In [58]:
ALPHA_INIT = [3.0, 3.0]     # αₖ at the start, one per cap
ALPHA_GOAL = [-3.0, -1.0]   # αₖ at the goal (a cap only turns one way: α_goal <= α_init)

rng = np.random.default_rng(0)
q_lower, q_upper = plant.GetPositionLowerLimits(), plant.GetPositionUpperLimits()


def random_free_configuration():
    """Uniform in the joint limits, until the arm configuration is in some free region."""
    while True:
        q = rng.uniform(q_lower, q_upper)
        if any(F.PointInSet(to_arm(q)) for F in free_sets):
            return q


q_init_astar = random_free_configuration()
q_goal_astar = random_free_configuration()
q_init_astar[idx_caps] = ALPHA_INIT
q_goal_astar[idx_caps] = ALPHA_GOAL

t_start = time.time()
astar_path, astar_info = reachability_astar(q_init_astar, q_goal_astar)
print(f"search time {time.time() - t_start:.3f}s, {astar_info['expanded']} nodes expanded")
print(f"start regions {astar_info['starts']}, goal regions {astar_info['goal_sets']}")

if astar_path is None:
    print("No plan found.")
else:
    n = count_actions(astar_path)
    print(f"plan: {n['grasps']} grasp(s) (strokes) {n['grasps_per_cap']} per cap, "
          f"{n['returns'] + n['releases']} ungrasp(s) "
          f"({n['returns']} return(s), {n['releases']} release(s)), {n['hops']} hop(s); "
          f"cost {astar_path[-1].g:.3f}\n")
    for node in astar_path:
        print(f"  {str(node.action):<24} -> {node}")

search time 0.039s, 555 nodes expanded
start regions [15, 18, 19], goal regions [8, 13, 14, 17, 24, 29, 30]
plan: 7 grasp(s) (strokes) [5, 2] per cap, 7 ungrasp(s) (5 return(s), 2 release(s)), 5 hop(s); cost 14.005

  ('start',)               -> F19  α1∈[+3.000, +3.000]  α2∈[+3.000, +3.000]  g=0.000
  ('transit', 19, 1)       -> F1   α1∈[+3.000, +3.000]  α2∈[+3.000, +3.000]  g=0.001
  ('enter', 1, 1, (0, 1))  -> C1   α1∈[+3.000, +3.000]  α2∈[+3.000, +3.000]  g=0.002 [entered]
  ('stroke', 1)            -> C1   α1∈[+1.626, +3.000]  α2∈[+3.000, +3.000]  g=1.002 [holding]
  ('return', 1)            -> C1   α1∈[+1.626, +3.000]  α2∈[+3.000, +3.000]  g=2.002 [released]
  ('stroke', 1)            -> C1   α1∈[+0.251, +3.000]  α2∈[+3.000, +3.000]  g=3.002 [holding]
  ('return', 1)            -> C1   α1∈[+0.251, +3.000]  α2∈[+3.000, +3.000]  g=4.002 [released]
  ('stroke', 1)            -> C1   α1∈[-1.123, +3.000]  α2∈[+3.000, +3.000]  g=5.002 [holding]
  ('return', 1)            -> C1   α1∈[-1.

### Step 6 — Place configurations along the chosen polytope sequence

A* returned *which* polytopes to go through and the exact sequence of grasp actions in each grasp visit
(e.g. `stroke, return, stroke, exit`), with the reachable α intervals. Now we pick actual configurations
(CASSR's second stage), without any optimisation solver, following that action sequence exactly:

1. **Allocate the turning.** Each stroke can turn at most its cap: `w_max − e_lo` for a stroke made straight
   after entering, `L` after a return. Walk the grasp visits backwards from α_goal: each visit turns
   `T = min(α_hi before the visit, α_after + Σ caps) − α_after` (lands exactly on α_init, since the search
   guaranteed α_goal was reachable). T is split among the visit's strokes by water-filling under the caps.
2. **Place points ("ends high" rule, so the exit is always reachable).**
   - transit F_i → F_j: the door (Chebyshev centre of F_i ∩ F_j);
   - enter: if the visit starts with a stroke, the point of F_i ∩ G at wrist `w_max − t1` (clipped to the
     entry range), so the stroke starts right there; if it starts with a return, the highest-wrist point;
   - stroke: wrist ↑ by t, cap −= t;
   - return: wrist ↓ to `min(current, w_max − t_next)`, cap still;
   - exit (release): the point of G ∩ F_j at wrist `min(current, highest exit wrist)` (wrist ↓ or still);
   - finally q_goal.

   Grasp points lie on the segment between G's lowest- and highest-wrist points (or between the extreme
   points of F_i ∩ G for entry/exit), so they stay inside G by convexity.

Each waypoint records the polytope that certifies the straight line *into* it. α goes in the cap slot,
so every waypoint is a full 10-D configuration.

**Several caps.** Step 1 runs cap by cap, over that cap's grasp visits only (they are the only moves that
change αₖ), from αₖ_goal back to αₖ_init. In step 2, each stroke turns its own visit's cap.

In [59]:
def point_at_wrist(comp, w):
    # Point of the grasp piece with wrist value w, on the segment between its extreme-wrist points.
    lo, hi = comp["chain"][0], comp["chain"][-1]
    s = (w - comp["w_min"]) / comp["L"]
    return lo + np.clip(s, 0.0, 1.0) * (hi - lo)


def point_in_link(link, w):
    # Point of (free region ∩ G) with wrist value w, between its lowest- and highest-wrist points.
    span = link["entry_w"] - link["exit_w"]
    s = 0.0 if span < 1e-12 else (w - link["exit_w"]) / span
    return link["exit"] + np.clip(s, 0.0, 1.0) * (link["entry"] - link["exit"])


def find_link(i, c, piece):
    return next(link for link in fc_links[(i, c)] if link["piece"] == piece)


def split_turn(T, caps):
    # Water-filling: split T as evenly as possible with amounts[i] <= caps[i].
    amounts, remaining = [0.0] * len(caps), T
    for rank, i in enumerate(sorted(range(len(caps)), key=lambda i: caps[i])):
        amounts[i] = min(caps[i], remaining / (len(caps) - rank))
        remaining -= amounts[i]
    assert remaining < 1e-6, f"cannot turn {T:.3f} with stroke caps {caps}"
    return amounts


def build_waypoints(path, q_init, q_goal):
    alpha_init = np.asarray(q_init, dtype=float)[idx_caps]
    alpha_goal = np.asarray(q_goal, dtype=float)[idx_caps]

    # 1. Split the path into grasp visits (their action lists) and allocate the turning (backward pass).
    visits = []
    for idx, node in enumerate(path):
        a = node.action
        if a[0] == "enter":
            assert len(components[a[2]]["pieces"]) == 1, "step 6 needs MERGE_PIECES = False"
            cap_k = components[a[2]]["cap"]
            visits.append(dict(comp=a[2], cap=cap_k, entry=find_link(a[1], a[2], a[3]), actions=[],
                               hi_before=path[idx - 1].alpha_hi[cap_k]))
        elif a[0] in ("stroke", "return"):
            visits[-1]["actions"].append(a[0])
        elif a[0] == "exit":
            visits[-1]["exit"] = find_link(a[2], a[1], a[3])

    for v in visits:
        comp = components[v["comp"]]
        v["caps"] = [comp["w_max"] - v["entry"]["exit_w"] if pos == 0 else comp["L"]
                     for pos, act in enumerate(v["actions"]) if act == "stroke"]

    for cap_k in range(K):                         # only cap k's visits change α_k
        alpha_after = alpha_goal[cap_k]
        for v in reversed([v for v in visits if v["cap"] == cap_k]):
            alpha_before = min(v["hi_before"], alpha_after + sum(v["caps"]))
            v["T"] = alpha_before - alpha_after
            v["amounts"] = split_turn(v["T"], v["caps"]) if v["caps"] else []
            alpha_after = alpha_before
        assert abs(alpha_after - alpha_init[cap_k]) < 1e-6, \
            f"cap {cap_k + 1}: turning allocation failed: {alpha_after} vs {alpha_init[cap_k]}"

    # 2. Place the waypoints: (nq-D q, label, certifying set) where the set contains the line into q.
    waypoints = [(np.asarray(q_init, dtype=float).copy(), "start", ("F", path[0].set_id))]
    alpha = alpha_init.copy()

    def add(x, label, cert):
        waypoints.append((to_full(x, alpha), label, cert))

    visit_iter = iter(visits)
    for node in path[1:]:
        a = node.action
        if a[0] == "transit":
            add(ff_door[(a[1], a[2])], "transit", ("F", a[1]))
        elif a[0] == "enter":
            v = next(visit_iter)
            comp, cert = components[v["comp"]], ("G", v["comp"])
            e_lo, e_hi = v["entry"]["exit_w"], v["entry"]["entry_w"]
            x_lo, x_hi = v["exit"]["exit_w"], v["exit"]["entry_w"]
            amounts = v["amounts"]
            if v["actions"] and v["actions"][0] == "stroke":
                c = float(np.clip(comp["w_max"] - amounts[0], max(e_lo, x_lo - v["T"]), e_hi))
            else:
                c = e_hi
            add(point_in_link(v["entry"], c), "enter grasp", ("F", a[1]))

            k = 0                                          # index of the next stroke
            for act in v["actions"]:
                if act == "stroke":
                    alpha[v["cap"]] -= amounts[k]
                    c += amounts[k]
                    k += 1
                    add(point_at_wrist(comp, c), "stroke (grasp)", cert)
                else:
                    t_next = amounts[k] if k < len(amounts) else 0.0
                    c = min(c, comp["w_max"] - t_next)
                    add(point_at_wrist(comp, c), "return (ungrasp)", cert)
            assert c >= x_lo - 1e-6, f"exit wrist {x_lo:.3f} not reachable from {c:.3f}"
            c = min(c, x_hi)
            label = "exit (release)" if v["amounts"] else "exit"
            add(point_in_link(v["exit"], c), label, cert)
        # "stroke", "return" and "exit" were already placed while executing the visit.

    add(to_arm(q_goal), "goal", ("F", path[-1].set_id))
    assert np.allclose(alpha, alpha_goal, atol=1e-6)
    return waypoints, visits

astar_waypoints, astar_visits = build_waypoints(astar_path, q_init_astar, q_goal_astar)
for v in astar_visits:
    print(f"grasp visit in C{v['comp']} (cap {v['cap'] + 1}): actions {v['actions']}, turns T = {v['T']:.3f} rad "
          f"as {np.round(v['amounts'], 3).tolist()}")
print(f"\n{len(astar_waypoints)} waypoints:")
for q, label, cert in astar_waypoints:
    caps_str = "  ".join(f"cap{k + 1} {q[i]:+.3f}" for k, i in enumerate(idx_caps))
    print(f"  {label:<18} in {cert[0]}{cert[1]:<3} wrist {q[idx_wrist]:+.3f}  {caps_str}")

grasp visit in C1 (cap 1): actions ['stroke', 'return', 'stroke', 'return', 'stroke', 'return', 'stroke', 'return', 'stroke'], turns T = 6.000 rad as [1.2, 1.2, 1.2, 1.2, 1.2]
grasp visit in C2 (cap 2): actions ['stroke', 'return', 'stroke'], turns T = 4.000 rad as [2.0, 2.0]

21 waypoints:
  start              in F19  wrist +0.083  cap1 +3.000  cap2 +3.000
  transit            in F19  wrist -0.084  cap1 +3.000  cap2 +3.000
  enter grasp        in F1   wrist -0.200  cap1 +3.000  cap2 +3.000
  stroke (grasp)     in G1   wrist +1.000  cap1 +1.800  cap2 +3.000
  return (ungrasp)   in G1   wrist -0.200  cap1 +1.800  cap2 +3.000
  stroke (grasp)     in G1   wrist +1.000  cap1 +0.600  cap2 +3.000
  return (ungrasp)   in G1   wrist -0.200  cap1 +0.600  cap2 +3.000
  stroke (grasp)     in G1   wrist +1.000  cap1 -0.600  cap2 +3.000
  return (ungrasp)   in G1   wrist -0.200  cap1 -0.600  cap2 +3.000
  stroke (grasp)     in G1   wrist +1.000  cap1 -1.800  cap2 +3.000
  return (ungrasp)   in G1  

#### 6b — Check every straight-line move
Each move must (1) have both endpoints in its certifying polytope (so the whole line is inside it) and
(2) respect the cap rule: inside a grasp piece of cap k, wrist ↑ turns cap k by exactly −Δwrist and leaves
the other caps alone; wrist ↓ keeps every cap; outside grasp no cap moves.

In [60]:
def certifying_set(cert):
    kind, idx = cert
    return free_sets[idx] if kind == "F" else pieces[components[idx]["pieces"][0]]["poly"]


all_ok = True
for (qa, _, _), (qb, label, cert) in zip(astar_waypoints[:-1], astar_waypoints[1:]):
    poly = certifying_set(cert)
    inside = poly.PointInSet(to_arm(qa), 1e-6) and poly.PointInSet(to_arm(qb), 1e-6)
    dw, dalpha = qb[idx_wrist] - qa[idx_wrist], qb[idx_caps] - qa[idx_caps]
    expected = np.zeros(K)
    if cert[0] == "G" and dw > TOL:
        expected[components[cert[1]]["cap"]] = -dw
    cap_ok = np.allclose(dalpha, expected, atol=1e-6)
    all_ok &= inside and cap_ok
    print(f"  {label:<20} {cert[0]}{cert[1]:<3} inside={inside!s:<5} cap_rule={cap_ok!s:<5} "
          f"Δwrist={dw:+.3f} Δα={np.round(dalpha, 3)}")
print("all moves valid:", all_ok)

  transit              F19  inside=True  cap_rule=True  Δwrist=-0.167 Δα=[0. 0.]
  enter grasp          F1   inside=True  cap_rule=True  Δwrist=-0.116 Δα=[0. 0.]
  stroke (grasp)       G1   inside=True  cap_rule=True  Δwrist=+1.200 Δα=[-1.2  0. ]
  return (ungrasp)     G1   inside=True  cap_rule=True  Δwrist=-1.200 Δα=[0. 0.]
  stroke (grasp)       G1   inside=True  cap_rule=True  Δwrist=+1.200 Δα=[-1.2  0. ]
  return (ungrasp)     G1   inside=True  cap_rule=True  Δwrist=-1.200 Δα=[0. 0.]
  stroke (grasp)       G1   inside=True  cap_rule=True  Δwrist=+1.200 Δα=[-1.2  0. ]
  return (ungrasp)     G1   inside=True  cap_rule=True  Δwrist=-1.200 Δα=[0. 0.]
  stroke (grasp)       G1   inside=True  cap_rule=True  Δwrist=+1.200 Δα=[-1.2  0. ]
  return (ungrasp)     G1   inside=True  cap_rule=True  Δwrist=-1.200 Δα=[0. 0.]
  stroke (grasp)       G1   inside=True  cap_rule=True  Δwrist=+1.200 Δα=[-1.2  0. ]
  exit (release)       G1   inside=True  cap_rule=True  Δwrist=+0.000 Δα=[0. 0.]
  transi

### Step 7 — Show the plan in Meshcat
Same playback as the brute force: straight-line interpolation between consecutive waypoints (the cap
value is interpolated too, which is exactly the α −= Δwrist rule on forward runs). The trajectory is
also stored as a `PiecewisePolynomial`, so it can be given to the slice viewer (section 5).

**Markers** (`meshcat_plan_markers.py`): green = start, orange = goal.
- Translucent ghosts of the gripper (its real meshes) at the start and goal configurations.
- On the cap: a green and an orange tick at the start and goal angles. The cap is at the start (goal)
  when its red line is on the green (orange) tick. The grey spiral arc and its arrowhead show
  which way and how far the cap turns, so a turn close to 2π is not confused with a small one.
- Around the wrist axis: the joint range (grey), the angles outside it (red), green and orange ticks
  at the start and goal wrist values, and a black needle along the finger opening that turns with
  the wrist. The needle staying inside the grey band shows that the plan respects the wrist limits.

With several caps, there is one set of cap markers per cap (`/plan_markers/cap1`, `/plan_markers/cap2`, ...).

In [61]:
from pydrake.trajectories import PiecewisePolynomial
import meshcat_plan_markers
import importlib
importlib.reload(meshcat_plan_markers)   # for iterative development in a notebook
from meshcat_plan_markers import PlanMarkers

def play_waypoints(waypoints, step=0.02, pause=0.02, markers=()):
    for (qa, _, _), (qb, label, _) in zip(waypoints[:-1], waypoints[1:]):
        n = max(2, int(np.ceil(np.linalg.norm(qb - qa) / step)))
        for s in np.linspace(0.0, 1.0, n):
            plant.SetPositions(plant_context, qa + s * (qb - qa))
            for m in markers:
                m.update(plant_context)
            diagram.ForcedPublish(diagram_context)
            time.sleep(pause)


Q = np.array([q for q, _, _ in astar_waypoints])
times = np.concatenate([[0.0], np.cumsum(np.linalg.norm(np.diff(Q, axis=0), axis=1))])
keep = np.concatenate([[True], np.diff(times) > 1e-9])        # drop zero-length moves
astar_traj = PiecewisePolynomial.FirstOrderHold(times[keep], Q[keep].T)
print(f"path length ({nq}-D, caps included): {times[-1]:.3f}")

wrist_joint = plant.GetJointByName("panda_joint7", panda_arm)
w_lo, w_hi = wrist_joint.position_lower_limits()[0], wrist_joint.position_upper_limits()[0]
print("wrist within its limits at every waypoint:",
      all(w_lo - 1e-9 <= q[idx_wrist] <= w_hi + 1e-9 for q, _, _ in astar_waypoints))

visualizer.meshcat_task_space.Delete("/plan_markers")
markers = [PlanMarkers(visualizer.meshcat_task_space, plant, visualizer.model_inspector, panda_hand,
                       wrist_joint=wrist_joint,
                       cap_joint=plant.GetJointByName("cap_to_base", cap_k),
                       prefix=f"/plan_markers/cap{k + 1}")
           for k, cap_k in enumerate(caps)]
for m in markers:
    m.set_start_goal(q_init_astar, q_goal_astar)

play_waypoints(astar_waypoints, markers=markers)

path length (11-D, caps included): 42.422
wrist within its limits at every waypoint: True


In [ ]:
# Optional: overlay the plan on the wrist/cap slice viewer (needs section 5's slice_viz)
if "slice_viz" in globals():
    slice_viz.set_trajectory(astar_traj)

## 5. C-space slice viewer

Interactive 2-D cross-section of the grasp/placement/c-free polytopes
(`cspace_slice_visualizer.py`) - reuses `visualizer` (this notebook's own
`CIrisPlantVisualizer`) so dragging any slider also moves the Meshcat robot
above. Defaults to the wrist / cap-1 plane (change `free_axes` to look at another
cap; the DOF the grasp polytopes
leave free - see `tc_space_investigation_summary.md` Part 2), seeded at
`q0` so the first render is non-degenerate.

For exploring the same polytopes *while this notebook's kernel is busy*
solving (section 4), use `cspace_explorer.ipynb` instead, in its own
kernel - this cell's sliders are frozen for as long as any cell above is
still running.

In [ ]:
from cspace_slice_visualizer import CSpaceSliceVisualizer

slice_viz = CSpaceSliceVisualizer.from_visualizer(
    visualizer,
    groups={
        "c-free": (planner.cs_free, "steelblue"),
        **{f"grasp cap {k + 1}": (G, color)
           for k, (G, color) in enumerate(zip(grasp_polytopes, ["seagreen", "orchid", "goldenrod", "teal"]))},
        "placement": (placement_polytope, "darkorange"),
    },
    q0=q0s[0],
    free_axes=(idx_wrist, idx_caps[0]),  # the wrist and cap 1 (DOF the grasp polytopes leave free)
)
slice_viz.show()
slice_viz.set_trajectory(astar_traj)